In [ ]:
import torch
import torch.nn as nn
import copy
import os
import json
from transformers import DistilBertModel,DistilBertConfig

In [ ]:
class QFormer(nn.Module):
    def __init__(self,bert_model = DistilBertModel , n_queries = 32, cross_every = 2, num_of_heads = 12 ):
        super().__init__()
        self.cross_every = cross_every
        self.n_queries = n_queries
        self.num_of_heads = num_of_heads

        # Save the config of the passed bert model to reconstruct it later
        self.bert_config = bert_model.config
        cfg: DistilBertConfig = self.bert_config
        self.hidden_size = cfg.hidden_size
        self.embeddings = copy.deepcopy(bert_model.embeddings)
        self.encoder_layers = nn.ModuleList([copy.deepcopy(layer) for layer in bert_model.transformer.layer])
        self.cross_blocks = nn.ModuleDict()

        for i in range(len(self.encoder_layers)):
            if(i % cross_every == cross_every-1):
                self.cross_blocks[str(i)] = nn.ModuleList(CrossAttentionBlock(self.hidden_size, self.num_of_heads))

        self.query_embeddings = nn.Parameter(torch.randn(1, n_queries, self.hidden_size))


    def save_pretrained(self, save_directory):
        """ Saves the Qformer model weights and config to a directory"""

        os.makedirs(save_directory, exist_ok=True)

        # Save config
        config = {
            "n_queries"   : self.n_queries,
            "cross_every" : self.cross_every,
            "num_heads"   : self.num_of_heads,
            "bert_config" : self.bert_config.to_dict()
        }

        with open(os.path.join(save_directory, "config.json"), "w") as f:
            json.dump(config, f, indent=2)

        # save the weights
        torch.save(self.state_dict(), os.path.join(save_directory, "pytorch_model.bin"))

    @classmethod
    def from_pretrained(cls, load_directory):

        """ Loads the Qformer model from a directory containing config.json and pytorch_model.bin files """

        # Load Config
        with open(os.path.join(load_directory, "config.json"), "r") as f:
            config = json.load(f)

        bert_config_dict = config.pop("bert_config")
        bert_config = DistilBertConfig.from_dict(bert_config_dict)

        # the weights will be overwritten by the load_state dict later
        bert_model = DistilBertModel(bert_config)
        model = cls(bert_model=bert_model, **config)
        
        #load weights
        state_dict_path = os.path.join(load_directory, "pytorch_model.bin")
        state_dict = torch.load(state_dict_path, map_location="cpu")
        model.load_state_dict(state_dict)

        return model
